# Causal mBERT on 6 pairs: Hindi, Spanish, French, Italian, German, Tagalog → English

In [ ]:
import os

import pandas as pd
from huggingface_hub import HfFileSystem

# On Kaggle: add HF_TOKEN under Add-ons > Secrets. Locally: read it from the repo-root .env
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except ImportError:
    from dotenv import find_dotenv, load_dotenv
    # Searches upward from the working directory, so the repo-root .env is found from kaggle/
    load_dotenv(find_dotenv(usecwd=True))
HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found - add it as a Kaggle Secret, or to the .env file at the repo root"

REPO_ID = "CodeSwitchGroup4/switchlingua-tagged-final"

In [ ]:
# Stream from the Hub into memory - nothing is written to disk
fs = HfFileSystem(token=HF_TOKEN)
with fs.open(f"datasets/{REPO_ID}/token_labels_word.parquet") as f:
    labels = pd.read_parquet(f)

print(labels.shape)
labels.groupby(["first_language", "second_language"]).size()

In [ ]:
# Readable name -> name used in the dataset (Tagalog is stored as "Philippines")
LANGS = {
    "Hindi": "Hindi",
    "Spanish": "Spanish",
    "French": "French",
    "Italian": "Italian",
    "German": "German",
    "Tagalog": "Philippines",
}
to_readable = {v: k for k, v in LANGS.items()}

# English as second language also excludes German-French
mask = labels["first_language"].isin(LANGS.values()) & (labels["second_language"] == "English")
df = labels[mask].reset_index(drop=True)
df["pair"] = df["first_language"].map(to_readable) + "-English"

print(df.shape)
df["pair"].value_counts()

In [ ]:
missing = set(LANGS) - set(df["pair"].str.removesuffix("-English"))
assert not missing, f"No rows for: {missing}"

pd.crosstab(df["pair"], df["split"], margins=True)

In [ ]:
row = df[df["pair"] == "Tagalog-English"].iloc[0]
pd.DataFrame({c: row[c] for c in ["tokens", "lang", "y_sw", "y_dur"]}).head(30)

## Labels in the spec's format

`sw`: 1 if the next word switches language, 0 if not, -1 ignored (punctuation, last word).
`dur`: 0 small (1–2), 1 medium (3–6), 2 large (7+), -1 where there is no switch or the burst is cut off before its class is known.

In [ ]:
DUR_CLASS = {"small": 0, "medium": 1, "large": 2}


def to_dur(y_sw, y_dur, dur_known):
    # Learn duration only at a switch whose class is certain
    return [DUR_CLASS[d] if s == 1 and k else -1 for s, d, k in zip(y_sw, y_dur, dur_known)]


df["sw"] = df["y_sw"].apply(list)
df["dur"] = [to_dur(*r) for r in zip(df["y_sw"], df["y_dur"], df["dur_known"])]

flat = df[["pair", "sw", "dur"]].explode(["sw", "dur"]).astype({"sw": int, "dur": int})
scored = flat[flat["sw"] != -1]
switches = flat[flat["sw"] == 1]
summary = pd.DataFrame({
    "switch_rate": scored.groupby("pair")["sw"].mean(),
    "small": switches.groupby("pair")["dur"].apply(lambda d: (d == 0).mean()),
    "medium": switches.groupby("pair")["dur"].apply(lambda d: (d == 1).mean()),
    "large": switches.groupby("pair")["dur"].apply(lambda d: (d == 2).mean()),
    "dur_masked": switches.groupby("pair")["dur"].apply(lambda d: (d == -1).mean()),
})
summary.round(3)

In [ ]:
import numpy as np


def f1_switch(y, p):
    # F1 of the SWITCH class (Anticipatory F1)
    y, p = np.asarray(y), np.asarray(p)
    tp = ((y == 1) & (p == 1)).sum()
    return 2 * tp / (2 * tp + ((y == 0) & (p == 1)).sum() + ((y == 1) & (p == 0)).sum()) if tp else 0.0


# Naive baselines: predict the next word's switch from the current word alone
words = df[["pair", "split", "tokens", "lang", "sw"]].explode(["tokens", "lang", "sw"])
words = words[words["sw"] != -1].astype({"sw": int})
words["word"] = words["tokens"].str.lower()

rows = []
for pair, g in words.groupby("pair"):
    train, val, test = (g[g["split"] == s] for s in ["train", "val", "test"])
    word_rate, prior = train.groupby("word")["sw"].mean(), train["sw"].mean()
    lang_rate = train.groupby("lang")["sw"].mean()
    score = lambda d: d["word"].map(word_rate).fillna(prior).to_numpy()
    cutoff = max(np.arange(0.05, 0.95, 0.05), key=lambda c: f1_switch(val["sw"], score(val) >= c))
    rows.append({
        "pair": pair,
        "always_switch": f1_switch(test["sw"], np.ones(len(test))),
        "current_lang": f1_switch(test["sw"], test["lang"].map(lang_rate) >= 0.5),
        "current_word": f1_switch(test["sw"], score(test) >= cutoff),
    })

baselines = pd.DataFrame(rows).set_index("pair")
baselines.loc["mean"] = baselines.mean()
baselines.loc["sigma"] = baselines.iloc[:-1].std()
baselines.round(3)

## mBERT tokenization

Each word's labels go on its **last** piece, so the prediction is made once the whole of word t has been read. Every other piece (and `[CLS]`, `[SEP]`) is -1 and ignored by the loss.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# mBERT has no curly quotes or long dashes, so "J’ai" or "it’s" would become [UNK]
ASCII = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-"})


def normalize(word):
    return word.translate(ASCII)


def align(word_ids, sw, dur):
    # Copy each word's labels to its last piece; everything else is -1
    sw_p, dur_p = [-1] * len(word_ids), [-1] * len(word_ids)
    for i, w in enumerate(word_ids):
        if w is not None and (i + 1 == len(word_ids) or word_ids[i + 1] != w):
            sw_p[i], dur_p[i] = sw[w], dur[w]
    return sw_p, dur_p


row = df[df["pair"] == "Tagalog-English"].iloc[0]
enc = tokenizer([normalize(w) for w in row["tokens"]], is_split_into_words=True)
sw_p, dur_p = align(enc.word_ids(), row["sw"], row["dur"])
pd.DataFrame({
    "piece": tokenizer.convert_ids_to_tokens(enc["input_ids"]),
    "word": enc.word_ids(),
    "sw": sw_p,
    "dur": dur_p,
}).head(30)

In [ ]:
encs = tokenizer([[normalize(w) for w in ws] for ws in df["tokens"]], is_split_into_words=True)
df["input_ids"] = encs["input_ids"]
df["sw_p"], df["dur_p"] = zip(*(align(encs.word_ids(i), sw, dur) for i, (sw, dur) in enumerate(zip(df["sw"], df["dur"]))))

n_pieces = df["input_ids"].str.len()
assert n_pieces.max() <= MAX_LEN, "some conversations need windows"

# Every word-level label must survive tokenization
count = lambda col, v: df[col].apply(lambda xs: sum(x == v for x in xs)).sum()
for v in (0, 1):
    assert count("sw", v) == count("sw_p", v), f"lost sw={v} labels"
assert count("dur", -1) - count("dur_p", -1) == sum(len(x) for x in df["dur"]) - n_pieces.sum(), "lost dur labels"

unk = df["input_ids"].apply(lambda ids: ids.count(tokenizer.unk_token_id))
pd.DataFrame({
    "pieces_per_word": n_pieces.groupby(df["pair"]).sum() / df["tokens"].str.len().groupby(df["pair"]).sum(),
    "unk_rate": unk.groupby(df["pair"]).sum() / n_pieces.groupby(df["pair"]).sum(),
    "max_pieces": n_pieces.groupby(df["pair"]).max(),
}).round(4)

## Causal mBERT (run on Kaggle)

Everything above runs locally; the cells below need torch and a GPU. On Kaggle: Settings > Accelerator > GPU, Settings > Internet on, and Add-ons > Secrets > `HF_TOKEN`. Then run the whole notebook top to bottom.

`SMOKE = True` trains briefly on a small sample to check the pipeline end to end; set it to `False` for the real run.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
SMOKE = True
BATCH_SIZE = 16
PAIRS = sorted(df["pair"].unique())
print(DEVICE)


class Conversations(Dataset):
    def __init__(self, frame):
        self.items = list(zip(frame["input_ids"], frame["sw_p"], frame["dur_p"], frame["pair"].map(PAIRS.index)))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    # Right-pad to the longest conversation; with a causal mask no real piece can see the padding
    n = max(len(ids) for ids, *_ in batch)
    pad = lambda xs, v: [list(x) + [v] * (n - len(x)) for x in xs]
    ids, sw, dur, pair = zip(*batch)
    return {
        "input_ids": torch.tensor(pad(ids, tokenizer.pad_token_id)),
        "attention_mask": torch.tensor(pad([[1] * len(x) for x in ids], 0)),
        "sw": torch.tensor(pad(sw, -1)),
        "dur": torch.tensor(pad(dur, -1)),
        "pair": torch.tensor(pair),
    }


def split_frame(name, per_pair):
    frame = df[df["split"] == name]
    return frame.groupby("pair").sample(per_pair, random_state=0) if SMOKE else frame


train_loader = DataLoader(Conversations(split_frame("train", 300)), BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(Conversations(split_frame("val", 50)), BATCH_SIZE, collate_fn=collate)
print(len(train_loader.dataset), "train /", len(val_loader.dataset), "val conversations")

batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items()}

In [ ]:
from torch import nn
from transformers import BertModel

LAMBDA_SW, LAMBDA_DUR = 1.0, 1.0


class CausalMBERT(nn.Module):
    def __init__(self, causal=True):
        super().__init__()
        # is_decoder=True gives a look-ahead mask: piece t attends only to pieces 1..t
        self.encoder = BertModel.from_pretrained(MODEL_NAME, is_decoder=causal, add_pooling_layer=False)
        hidden = self.encoder.config.hidden_size
        self.switch_head = nn.Linear(hidden, 2)
        self.dur_head = nn.Linear(hidden, 3)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
        return self.switch_head(h), self.dur_head(h)


ce = nn.CrossEntropyLoss(ignore_index=-1)


def multitask_loss(sw_logits, dur_logits, sw, dur):
    loss_sw = ce(sw_logits.flatten(0, 1), sw.flatten())
    # A batch with no known-duration switch would give NaN, so count it as 0
    has_dur = (dur != -1).any()
    loss_dur = ce(dur_logits.flatten(0, 1), dur.flatten()) if has_dur else sw_logits.new_zeros(())
    return LAMBDA_SW * loss_sw + LAMBDA_DUR * loss_dur, loss_sw, loss_dur


model = CausalMBERT().to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

In [ ]:
@torch.no_grad()
def prefix_change(encoder, ids, k):
    # Largest change in the outputs for pieces 0..k when every piece after k is replaced
    future = ids.clone()
    future[0, k + 1:] = torch.randint(1000, tokenizer.vocab_size, (ids.shape[1] - k - 1,), device=ids.device)
    out = lambda x: encoder(input_ids=x, use_cache=False).last_hidden_state[0, : k + 1]
    return (out(ids) - out(future)).abs().max().item()


ids = torch.tensor([val_loader.dataset[0][0]], device=DEVICE)
k = ids.shape[1] // 2

model.eval()
causal_change = prefix_change(model.encoder, ids, k)
bidirectional = BertModel.from_pretrained(MODEL_NAME, add_pooling_layer=False).to(DEVICE).eval()
bidirectional_change = prefix_change(bidirectional, ids, k)
del bidirectional

print(f"change in outputs 0..{k} after replacing pieces {k + 1}..{ids.shape[1] - 1}:")
print(f"  causal mBERT:        {causal_change:.2e}")
print(f"  bidirectional mBERT: {bidirectional_change:.2e}")
assert causal_change < 1e-4, "future pieces leak into the prefix"

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sw_true, sw_pred, dur_true, dur_pred, sw_pair, dur_pair = [], [], [], [], [], []
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        sw_logits, dur_logits = model(b["input_ids"], b["attention_mask"])
        pair = b["pair"][:, None].expand_as(b["sw"])
        s, d = b["sw"] != -1, b["dur"] != -1
        sw_true.append(b["sw"][s]); sw_pred.append(sw_logits.argmax(-1)[s]); sw_pair.append(pair[s])
        dur_true.append(b["dur"][d]); dur_pred.append(dur_logits.argmax(-1)[d]); dur_pair.append(pair[d])
    sw_true, sw_pred, sw_pair, dur_true, dur_pred, dur_pair = (
        torch.cat(x).cpu().numpy() for x in (sw_true, sw_pred, sw_pair, dur_true, dur_pred, dur_pair))

    rows = []
    for i, pair in enumerate(PAIRS):
        s, d = sw_pair == i, dur_pair == i
        rows.append({
            "pair": pair,
            "anticipatory_f1": f1_switch(sw_true[s], sw_pred[s]),
            "duration_acc": (dur_true[d] == dur_pred[d]).mean(),
            # Naive word lookup, scored on the test split
            "naive_f1_test": baselines.loc[pair, "current_word"],
        })
    results = pd.DataFrame(rows).set_index("pair")
    results.loc["mean"] = results.mean()
    results.loc["sigma"] = results.iloc[:-1].std()
    return results

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 1 if SMOKE else 3
LR = 2e-5
use_amp = DEVICE == "cuda"

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=use_amp)

history = []
for epoch in range(EPOCHS):
    model.train()
    for step, b in enumerate(train_loader):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            sw_logits, dur_logits = model(b["input_ids"], b["attention_mask"])
        loss, loss_sw, loss_dur = multitask_loss(sw_logits.float(), dur_logits.float(), b["sw"], b["dur"])

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        history.append({"epoch": epoch, "step": step, "loss": loss.item(), "loss_sw": loss_sw.item(), "loss_dur": loss_dur.item()})
        if step % 50 == 0 or step == len(train_loader) - 1:
            recent = pd.DataFrame(history[-50:])[["loss", "loss_sw", "loss_dur"]].mean()
            print(f"epoch {epoch} step {step}/{len(train_loader)}  " + "  ".join(f"{k} {v:.3f}" for k, v in recent.items()))

    print(evaluate(model, val_loader).round(3))

history = pd.DataFrame(history)
if not SMOKE:
    torch.save(model.state_dict(), "/kaggle/working/causal_mbert.pt")
    history.to_csv("/kaggle/working/loss_log.csv", index=False)